# Libreta asociada al modulo de lectura, limpieza y validacion

## Librerias

In [1]:
import pandas as pd
import re
import numpy as np

In [ ]:
np.nan == pd.NA

## Funciones

Hay que documentar que hace cada funcion. Para que te hagas una idea de como deberia estar documentada, voy a plantear algunas funciones que o complementan o de plano sustituyen algunas funciones tuyas.

### Funcion: extaer_numero

Falta documentar esta funcion.

- Que hace?
- Que significa la expresion regular `r'-?\d+[\d,]*\.?\d*'`?

Por lo que yo puedo observar, esta funcion recibe un valor (me imagino idealmente numerico), y hace lo siguiente:

- Si valor es nulo, devuelve None
- Si en el valor hay una incidencia de la siguiente expresion regular `r'-?\d+[\d,]*\.?\d*'` (no se que signifique dicha expresion), entonces usando .group() devuelve  todo el match string; despues remueve las comas.
- Si en el valor no hay ninguna inciencia de la expresion regular antes descita, devuelve `None`.

No se que tan buena sea la funcion porque no se que evalua la expresion regualar, por favor se explicito en como funciona.

In [34]:
# Función para extraer número

def extraer_numero(valor):
    if pd.isna(valor):
        return None
    
    match = re.search(r'-?\d+[\d,]*\.?\d*', str(valor))
    if match:
        return float(match.group().replace(",", ""))
    return None


### Funcion: normalizar_nombre_cuenta

Falta documentar esta funcion:

- Que hace?
- Que formato estas siguiendo para normalizar los nombres?

Por lo que puedo observar, recibe un string. Si el string es igual a `None`, entonces retorna `None`. Si no es nulo haces lo siguiente:

- Remover trailing and leading espacios (i.e. " ").
- Remover la siguiente lista de substrings del string ["(-)", "( - )", "(Neto)", "(Netos)", "(netos)"]
- De nuevo remover trailing and leading espacios.
- Remplazar multples espacios en blanco contiguos con uno solo
- remplazar espacios con guiones bajos

**Problemas con esta funcion!!!**

Hay varios problemas claros con esta funcion:

1. En tu condicional evaluas si texto es None. Esto esta incompleto. Me imagino que texto lo obtienes leyendo las entradas del dataframe. Si esto es asi, entonces puede haber varios tipos de null values. Para profundizar al respecto puedes leer [esto](https://pandas.pydata.org/docs/user_guide/missing_data.html). Resumiendolo, tenemos los siguientes tipos de null values en pandas:
    - `np.nan`
    - `NaT`
    - `NA`
    - `None`

Ahorita tu solo evaluas `None`, pero los demas van a pasar como verdaderos, y pues realmente tambien son nulos. Para evitar esto, puedes usar el metodo `pd.isna(texto)`.

2. El estandar bajo el cual normalizas no es muy robusto. Solo remueves espacios en blanco, varios prefijos, espacios contiguos, y sustituyes espacios con guines bajos. Te falta asegurarte que no haya caracteres especiales, que no haya numeros, que todos esten en minuscula, etc. Tambien, idealmente, habria que eliminar todos los acentos. Esto es, si el texto es "Ávatár", se me ocurre habria que transformarlo a "avatar". Esto implica mapear las 'Á' con 'a', etc.


In [50]:
def normalizar_nombre_cuenta(texto):
    
    if pd.isna(texto):
        return None
    
    texto = str(texto).strip()
    
    texto = texto.replace("(-)", "")
    texto = texto.replace("( - )", "")
    texto = texto.replace("(Neto)", "")
    texto = texto.replace("(Netos)", "")
    texto = texto.replace("(neto)", "")
    texto = texto.strip()
    
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.replace(" ", "_")
    #texto = texto.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    #texto = texto.replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "Ú").replace("ü", "u").replace("Ü", "U")

    
    return texto

Mi recomendacion:

- Quitar espacios en blanco de las orillas
- Quitar prefix "( - )" o "(-)"
- Quitar acentos y dieresis
- Poner todo en mayusculas
- Sustituir espacios en blanco por guiones bajos
- Remover todo tipo de caracter que no sea una letra mayuscula de la A a la Z o un guion bajo
- Remover multiples guiones bajos contiguos

De esta manera, una cuenta como '*(-) Estimación para Castigos*', quedara como *ESTIMACION_PARA_CASTIGOS*.

A continuacion planteo dicha funcion:

In [42]:
def normalizar_nombre_cuenta_v2(texto: any) -> str | None:    # Aqui estamos aplicando type hint. Le estamos diciendo a python que el argumento de entrada 'texto' puede ser cualquier
                                                              # cosa, y que la funcion devuelve un string o None.
    if pd.isna(texto):    # Asi evaluamos si texto es None, np.nan, NA o NaT
        return None

    # Primero removemos espacios en los extremos, y los prefijos mencionados.
    texto_normalizado = str(texto).strip().removeprefix("(-)").removeprefix("( - )")

    # Ahora removemos todos los acentos y dieresis
    texto_normalizado = texto_normalizado.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    texto_normalizado = texto_normalizado.replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "Ú").replace("ü", "u").replace("Ü", "U")

    # Ahora ponemos todo en mayusculas, sutituimos espacios en blanco por guiones bajos, removemos todo tipo de caracter menos una letra mayuscula de la A a la Z o un guion bajo,
    # removemos multiples guiones bajos contiguos, y removemos guiones bajos en los extremos
    texto_normalizado = re.sub(r"[^A-Z_ñÑ]", "", texto_normalizado.upper().replace(" ", "_"))
    texto_normalizado = re.sub(r"_+", "_", texto_normalizado).strip("_")

    return texto_normalizado

test_string = "(-) Estimac/*123ióñ para Casti123goÑs aijsd890aiousehff 23487289374 ¿'¿{+ñ123}"

display(normalizar_nombre_cuenta_v2(test_string))

'ESTIMACIOÑ_PARA_CASTIGOÑS_AIJSDAIOUSEHFF_Ñ'

### Funciones: construir_diccionario_balance y construir_diccionario_resultados

Falta documentar las funciones:

- Que hacen?

In [81]:
def construir_diccionario_balance(path_csv, cuentas_objetivo):
    df = pd.read_csv(path_csv, encoding="latin1", sep=None, engine="python", on_bad_lines="skip")
    
    resultado = {}
    
    for cuenta in cuentas_objetivo:
        llave = normalizar_nombre_cuenta(cuenta)
        encontrado = False
        
        nueva_llave = llave.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace("ñ", "n").replace("Ñ", "N")
        nueva_llave = nueva_llave.replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "Ú").replace("ü", "u").replace("Ü", "U")

        for _, row in df.iterrows():
            for celda in row.values:
                
                if not pd.isna(celda):
                    nueva_celda = normalizar_nombre_cuenta(celda)
                    nueva_celda = nueva_celda.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace("ñ", "n").replace("Ñ", "N")
                    nueva_celda = nueva_celda.replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "Ú").replace("ü", "u").replace("Ü", "U")

                    if  nueva_celda == nueva_llave:
                        resultado[llave] = 1
                        encontrado = True
                        # for valor in row:
                        #     numero = extraer_numero(valor)
                        #     if numero is not None:
                        #         resultado[llave] = numero
                        #         encontrado = True
                        #         break
                        break
            
            if encontrado:
                break
        
        if not encontrado:
            resultado[llave] = None
    
    return resultado

def construir_diccionario_resultados(path_csv, cuentas_objetivo):
    df = pd.read_csv(path_csv, encoding="latin1", sep=None, engine="python", on_bad_lines="skip")
    
    resultado = {}
    
    for cuenta in cuentas_objetivo:
        llave = normalizar_nombre_cuenta(cuenta)
        encontrado = False
        
        for _, row in df.iterrows():
            for celda in row:
                
                if normalizar_nombre_cuenta(celda) == normalizar_nombre_cuenta(cuenta):
                    
                    for valor in row:
                        numero = extraer_numero(valor)
                        if numero is not None:
                            resultado[llave] = numero
                            encontrado = True
                            break
                    break
            
            if encontrado:
                break
        
        if not encontrado:
            resultado[llave] = None
    
    return resultado

In [75]:
def validar_balance(diccionario):
    
    errores = []
    
    if None in diccionario.values():
        lista_cuentas_faltantes = [key for key, value in diccionario.items() if value == None]
        string_cuentas_faltantes = "\n".join(lista_cuentas_faltantes)
        errores.append("Las siguientes cuentas no se encuentran en el .csv del balance general: ")
        return (False,"Las siguientes cuentas no se encuentran en el .csv del balance general:\n" + string_cuentas_faltantes)

    activos = diccionario.get(normalizar_nombre_cuenta("Suma del Activo"), 0)
    pasivos = diccionario.get(normalizar_nombre_cuenta("Suma del Pasivo"), 0)
    capital = diccionario.get(normalizar_nombre_cuenta("Suma del Capital"), 0)
    
    if activos != pasivos + capital:
        errores.append("Suma del Activo ≠ Suma del Pasivo + Capital")

    if len(errores) == 0:
        return (True, "")
    else:
        return (False, "; ".join(errores))

def validar_resultados(diccionario):
    errores = []
    
    if None in diccionario.values():
        lista_cuentas_faltantes = [key for key, value in diccionario.items() if value == None]
        string_cuentas_faltantes = "\n".join(lista_cuentas_faltantes)
        errores.append("Las siguientes cuentas no se encuentran en el .csv del estado de resultados: ")
        return (False,"Las siguientes cuentas no se encuentran en el .csv del estado de resultados:\n" + string_cuentas_faltantes)

    emitidas = diccionario.get(normalizar_nombre_cuenta("Emitidas"), 0)
    cedidas = diccionario.get(normalizar_nombre_cuenta("Cedidas"), 0)
    a = diccionario.get(normalizar_nombre_cuenta("Incremento Neto de la Reserva de Riesgos en Curso y de Fianzas en Vigor"), 0)
    b = diccionario.get(normalizar_nombre_cuenta("Costo Neto de Adquisición"), 0)
    c = diccionario.get(normalizar_nombre_cuenta("Costo Neto de Siniestralidad, Reclamaciones y Otras Obligaciones Pendientes de Cumplir"), 0)
    d = diccionario.get(normalizar_nombre_cuenta("Incremento Neto de Otras Reservas Técnicas"), 0)
    e = diccionario.get(normalizar_nombre_cuenta("Resultado de Operaciones Análogas y Conexas"), 0)
    f = diccionario.get(normalizar_nombre_cuenta("Gastos de Operación Netos"), 0)
    g = diccionario.get(normalizar_nombre_cuenta("Resultado Integral de Financiamiento"), 0)
    h = diccionario.get(normalizar_nombre_cuenta("Provisión para el pago del Impuestos a la Utilidad"), 0)
    i = diccionario.get(normalizar_nombre_cuenta("Utilidad (Pérdida) antes de Operaciones Discontinuadas"), 0)
    
    if abs(emitidas - (cedidas + a + b + c + d - e + f - g + h + i)) > 0.00001:
        errores.append("Las cuentas no dan como suma las primas emitidas")
     
    return (len(errores) == 0, "; ".join(errores))


In [76]:
def procesar_estados(balance_path, resultados_path):
    
    cuentas_balance = [
        "Inversiones",
        "Valores y Operaciones con Productos Derivados",
        "Valores",
        "Gubernamentales",
        "Empresas Privadas. Tasa Conocida",
        "Empresas Privadas. Renta Variable",
        "Extranjeros",
        "Dividendos por Cobrar sobre Títulos de Capital",
        "(-) Deterioro de Valores",
        "Inversiones en Valores dados en Préstamo",
        "Valores Restringidos",
        "Operaciones con Productos Derivados",
        "Deudor por Reporto",
        "Cartera de Crédito (Neto)",
        "Cartera de Crédito Vigente",
        "Cartera de Crédito Vencida",
        "(-) Estimaciones Preventivas por Riesgo Crediticio",
        "Inmuebles (Neto)",
        "Inversiones para Obligaciones Laborales",
        "Disponibilidad",
        "Caja y Bancos",
        "Deudores",
        "Por Primas",
        "Deudor por Prima por Subsidio Daños",
        "Adeudos a cargo de Dependencias y Entidades de la Administración Pública Federal",
        "Primas por Cobrar de Fianzas Expedidas",
        "Agentes y Ajustadores",
        "Documentos por Cobrar",
        "Deudores por Responsabilidades",
        "Otros",
        "(-) Estimación para Castigos",
        "Reaseguradores y Reafianzadores (Neto)",
        "Instituciones de Seguros y Fianzas",
        "Depósitos Retenidos",
        "Importes Recuperables de Reaseguro",
        "(-) Estimación preventiva de riesgos crediticios de  Reaseguradores Extranjeros",
        "Intermediarios de Reaseguro y Reafianzamiento",
        "(-) Estimación para Castigos",
        "Inversiones Permanentes",
        "Subsidiarias",
        "Asociadas",
        "Otras Inversiones Permanentes",
        "Otros Activos",
        "Mobiliario y Equipo (Neto)",
        "Activos Adjudicados (Neto)",
        "Diversos",
        "Activos Intangibles Amortizables (Netos)",
        "Activos Intangibles de larga duración (Netos)",
        "Suma del Activo",
        "Reservas Técnicas",
        "De Riesgos en Curso",
        "Seguros de Vida",
        "Seguros de Accidentes y Enfermedades",
        "Seguros de Daños",
        "Reafianzamiento Tomado",
        "De Fianzas en Vigor",
        "Reserva para Obligaciones Pendientes de Cumplir",
        "Por Pólizas Vencidas y Siniestros Ocurridos pendientes de Pago",
        "Por Siniestros Ocurridos y No Reportados y Gastos de Ajuste Asignados a los Siniestros",
        "Por Fondos en Administración",
        "Por Primas en Depósito",
        "Reserva de Contingencia",
        "Reserva para Seguros Especializados",
        "Reserva de Riesgos Catastróficos",
        "Reserva para Obligaciones Laborales",
        "Acreedores",
        "Agentes y Ajustadores",
        "Fondos en Administración de Pérdidas",
        "Acreedores por Responsabilidades de Fianzas por Pasivos Constituidos",
        "Diversos",
        "Reaseguradores y Reafianzadores",
        "Instituciones de Seguros y Fianzas",
        "Depósitos Retenidos",
        "Otras Participaciones",
        "Intermediarios de Reaseguro y Reafianzamiento",
        "Operaciones con Productos Derivados. Valor Razonable (parte pasiva) al momento de la adquisición",
        "Financiamientos Obtenidos",
        "Emisión de Deuda",
        "Por Obligaciones Subordinadas No Susceptibles de Convertirse en Acciones",
        "Otros Títulos de Crédito",
        "Contratos de Reaseguro Financiero",
        "Otros Pasivos",
        "Provisión para la Participación de los Trabajadores en la Utilidad",
        "Provisión para el Pago de Impuestos",
        "Otras Obligaciones",
        "Créditos Diferidos",
        "Suma del Pasivo",
        "Capital Contable",
        "Capital Contribuido",
        "Capital o Fondo Social Pagado",
        "Capital o Fondo Social",
        "(-) Capital o Fondo Social No Suscrito",
        "(-) Capital o Fondo Social No Exhibido",
        "(-) Acciones Propias Recompradas",
        "Obligaciones Subordinadas de Conversión Obligatoria a Capital",
        "Capital Ganado",
        "Reservas",
        "Legal",
        "Para Adquisición de Acciones Propias",
        "Otras",
        "Superávit por Valuación",
        "Inversiones Permanentes",
        "Resultados o Remanentes de Ejercicios Anteriores",
        "Resultado o Remanente del Ejercicio",
        "Resultado por Tenencia de Activos No Monetarios",
        "Remediciones por Beneficios Definidos a los Empleados",
        "Participación Controladora",
        "Participación No Controladora",
        "Suma del Capital",
        "Suma del Pasivo y Capital"
    ]

    cuentas_resultados = [
        "Emitidas",
        "Cedidas",
        "De Retención",
        "Incremento Neto de la Reserva de Riesgos en Curso y de Fianzas en Vigor",
        "Primas de Retención Devengadas",
        "Costo Neto de Adquisición",
        "Comisiones a Agentes",
        "Compensaciones Adicionales a Agentes",
        "Comisiones por Reaseguro y Reafianzamiento Tomado",
        "Comisiones por Reaseguro y Reafianzamiento Cedido",
        "Cobertura de Exceso de Pérdida",
        "Otros",
        "Costo Neto de Siniestralidad, Reclamaciones y Otras Obligaciones Pendientes de Cumplir",
        "Siniestralidad y Otras Obligaciones Pendientes de Cumplir",
        "Siniestralidad Recuperada del Reaseguro No Proporcional",
        "Reclamaciones",
        "Utilidad (Pérdida) Técnica",
        "Incremento Neto de Otras Reservas Técnicas",
        "Reserva para Riesgos Catastróficos",
        "Reserva para Seguros Especializados",
        "Reserva de Contingencia",
        "Otras Reservas",
        "Resultado de Operaciones Análogas y Conexas",
        "Utilidad (Pérdida) Bruta",
        "Gastos de Operación Netos",
        "Gastos Administrativos y Operativos",
        "Remuneraciones y Prestaciones al Personal",
        "Depreciaciones y Amortizaciones",
        "Utilidad (Pérdida) de la Operación",
        "Resultado Integral de Financiamiento",
        "De Inversiones",
        "Por Venta de Inversiones",
        "Por Valuación de Inversiones",
        "Por Recargo sobre Primas",
        "Por Emisión de Instrumentos de Deuda",
        "Por Reaseguro Financiero",
        "Intereses por créditos",
        "Castigos preventivos por importes recuperables de reaseguro",
        "Castigos preventivos por riesgos crediticios",
        "Otros",
        "Resultado Cambiario",
        "Resultado por Posición Monetaria",
        "Participación en el Resultado de Inversiones Permanentes",
        "Utilidad (Pérdida) antes de Impuestos a la Utilidad",
        "Provisión para el pago del Impuestos a la Utilidad",
        "Utilidad (Pérdida) antes de Operaciones Discontinuadas",
        "Operaciones Discontinuadas",
        "Utilidad (Pérdida) del Ejercicio",
        "Participación Controladora",
        "Participación No Controladora"
    ]

    balance_dict = construir_diccionario_balance(balance_path, cuentas_balance.copy())
    resultados_dict = construir_diccionario_resultados(resultados_path, cuentas_resultados)
    
    #val_balance = validar_balance(balance_dict)
    #val_resultados = validar_resultados(resultados_dict)
    
    #validaciones = {"BG": val_balance, "ER": val_resultados}
    
    return balance_dict, resultados_dict #, validaciones

In [80]:
bg_dict, er_dict = procesar_estados("INPUTS/Balance_general.csv", "INPUTS/Estado_de_resultados.csv")

val_balance = validar_balance(bg_dict)
val_resultados = validar_resultados(er_dict)

display(val_balance)
display(val_resultados)
display(normalizar_nombre_cuenta("Dividendos por Cobrar sobre Títulos de Capital"))

(False,
 'Las siguientes cuentas no se encuentran en el .csv del balance general:\nDividendos_por_Cobrar_sobre_Títulos_de_Capital\nInversiones_en_Valores_dados_en_Préstamo\nCartera_de_Crédito\nCartera_de_Crédito_Vigente\nCartera_de_Crédito_Vencida\nDeudor_por_Prima_por_Subsidio_Daños\nAdeudos_a_cargo_de_Dependencias_y_Entidades_de_la_Administración_Pública_Federal\nEstimación_para_Castigos\nDepósitos_Retenidos\nEstimación_preventiva_de_riesgos_crediticios_de_Reaseguradores_Extranjeros\nActivos_Intangibles_de_larga_duración\nReservas_Técnicas\nSeguros_de_Daños\nPor_Pólizas_Vencidas_y_Siniestros_Ocurridos_pendientes_de_Pago\nPor_Fondos_en_Administración\nPor_Primas_en_Depósito\nReserva_de_Riesgos_Catastróficos\nFondos_en_Administración_de_Pérdidas\nOperaciones_con_Productos_Derivados._Valor_Razonable_(parte_pasiva)_al_momento_de_la_adquisición\nEmisión_de_Deuda\nOtros_Títulos_de_Crédito\nProvisión_para_la_Participación_de_los_Trabajadores_en_la_Utilidad\nProvisión_para_el_Pago_de_Impuest

(False,
 'Las siguientes cuentas no se encuentran en el .csv del estado de resultados:\nDe_Retención\nPrimas_de_Retención_Devengadas\nCosto_Neto_de_Adquisición\nCobertura_de_Exceso_de_Pérdida\nUtilidad_(Pérdida)_Técnica\nIncremento_Neto_de_Otras_Reservas_Técnicas\nReserva_para_Riesgos_Catastróficos\nResultado_de_Operaciones_Análogas_y_Conexas\nUtilidad_(Pérdida)_Bruta\nGastos_de_Operación_Netos\nUtilidad_(Pérdida)_de_la_Operación\nPor_Valuación_de_Inversiones\nPor_Emisión_de_Instrumentos_de_Deuda\nIntereses_por_créditos\nResultado_por_Posición_Monetaria\nParticipación_en_el_Resultado_de_Inversiones_Permanentes\nUtilidad_(Pérdida)_antes_de_Impuestos_a_la_Utilidad\nProvisión_para_el_pago_del_Impuestos_a_la_Utilidad\nUtilidad_(Pérdida)_antes_de_Operaciones_Discontinuadas\nUtilidad_(Pérdida)_del_Ejercicio\nParticipación_Controladora\nParticipación_No_Controladora')

'Dividendos_por_Cobrar_sobre_Títulos_de_Capital'

In [ ]:
if __name__ == "__main__":
    
    balance_file = r"C:/Users/Diego/OneDrive/Documentos/Actuaría UANL 6th Semester/Contabilidad de Seguros/Prueba/Balance_general1.csv"
    resultados_file = r"C:/Users/Diego/OneDrive/Documentos/Actuaría UANL 6th Semester/Contabilidad de Seguros/Prueba/Estado_de_resultados.csv"
    
    balance, resultados, validaciones = procesar_estados(balance_file, resultados_file)
    
print("\nBalance General:")
for k, v in balance.items():
    print(f"{k}: {v}")
    
print("\nEstado de Resultados:")
for k, v in resultados.items():
    print(f"{k}: {v}")
    
print("\nValidaciones:")
for estado, (ok, error) in validaciones.items():
    print(f"{estado}: {'✔️' if ok else '❌'} {error}")

In [ ]:
for k, v in balance.items():
    if v is None:
        print("NO ENCONTRADA:", k)